In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Importing required libraries
import sqlite3
import numpy as np
import pandas as pd
import seaborn as sns
import pandasql as ps
import matplotlib.pyplot as plt
import os
from datetime import datetime
from IPython.core.magic import register_line_magic
from sqlalchemy import create_engine
import sqlalchemy

In [ ]:
#!pip install jupysql

### SQLITE CONNECTION FUNCTION

In [ ]:
def sqliteConnections():
    try:        
        sqlitedb_path = "/kaggle/input/airlines-dataset/travel.sqlite"
        with sqlite3.connect(sqlitedb_path) as sqliteConn:
            engine = create_engine(f"sqlite:////{sqlitedb_path}", echo=True) 
            print("✅ Connection Successful")
            return sqliteConn, engine  
        
    except sqlite3.Error as e:
        print(f"❌ Error Connecting to .db: {e}")
        return None, None

In [ ]:
# CONNECTIONS
sqlite_conn, engine = sqliteConnections()

In [ ]:
%load_ext sql

In [ ]:
%sql sqlite:////kaggle/input/airlines-dataset/travel.sqlite

In [ ]:
#sqlite_conn=sqlite3.connect("/kaggle/input/airlines-dataset/travel.sqlite")

### PROGRAMMATIC ASSESSMENT

- SQL tables

In [ ]:
%%sql
SELECT name FROM sqlite_schema WHERE type ='table';

#### Aircrafts Dataset

In [ ]:
%%sql 
SELECT * 
FROM aircrafts_data 
LIMIT 10;

- Extracting the english name from model

In [ ]:
%%sql
SELECT aircraft_code, 
    json_extract(model, '$.en') AS aircraft_model, 
    range
FROM aircrafts_data
LIMIT 10;

#### Airports Dataset
- Extracting english airport name and city name

In [ ]:
%%sql
SELECT airport_code, 
        json_extract(airport_name, '$.en') AS airport_name, 
        json_extract(city, '$.en') AS city, 
        coordinates, timezone
FROM airports_data 
LIMIT 5;

#### Boarding-passes Dataset

In [ ]:
%%sql
# load boarding-passes data
SELECT *
FROM boarding_passes 
LIMIT 5;

- Checking duplicates

In [ ]:
%%sql
# check for duplicate ticket numbers
SELECT 
        COUNT(ticket_no) AS total, 
        COUNT(DISTINCT ticket_no) AS unique_tickets,
        COUNT(ticket_no) - COUNT(DISTINCT ticket_no) AS dup_tickets
FROM boarding_passes;

#### Bookings Dataset

In [ ]:
%%sql
# load bookings data
SELECT *
FROM bookings 
LIMIT 5;

- Checking duplicates

In [ ]:
%%sql
# check for duplicate bookings references
SELECT 
        COUNT(book_ref) AS total, 
        COUNT(DISTINCT book_ref) unique_ref,
        COUNT(book_ref) - COUNT(DISTINCT book_ref) AS dup_ref
FROM bookings;

#### Flights Dataset

In [ ]:
%%sql
# load flights data
SELECT *
FROM flights 
LIMIT 5;

- Filtering missing data

In [ ]:
%%sql
# load flights data
SELECT *
FROM flights
WHERE actual_departure != "\N" OR actual_arrival != "\N"
LIMIT 5;

#### Seats Dataset

In [ ]:
%%sql
# load seats descriptions
SELECT *
FROM seats 
LIMIT 5;

#### Ticket Flights Dataset

In [ ]:
%%sql
# load ticket_flights data
SELECT *
FROM ticket_flights 
LIMIT 5;

#### Tickets Dataset

In [ ]:
%%sql
# load tickets data
SELECT *
FROM tickets
LIMIT 5;

### ANALYSIS AND INSIGHTS

#### 1.  Top 10 Airports with the Most Delays

In [ ]:
%%sql
WITH flight_data AS (
       SELECT 
                f.flight_id,
                f.flight_no,
                datetime(substr(f.scheduled_departure, 1, 19)) AS scheduled_departure,
                datetime(substr(f.actual_departure, 1, 19)) AS actual_departure,
                datetime(substr(f.scheduled_arrival, 1, 19)) AS scheduled_arrival,
                datetime(substr(f.actual_arrival, 1, 19)) AS actual_arrival,
                f.departure_airport,
                f.arrival_airport,
                f.status,
                json_extract(ap1.airport_name, '$.en') AS departure_airport_name,
                json_extract(ap1.city, '$.en') AS departure_city,
                ap1.coordinates AS departure_city_coordinates,
                ap1.timezone AS departure_timezone,
                json_extract(ap2.airport_name, '$.en') AS arrival_airport_name,
                json_extract(ap2.city, '$.en') AS arrival_city,
                ap2.coordinates AS arrival_city_coordinates,
                ap2.timezone AS arrival_timezone
        FROM flights f
        JOIN airports_data ap1
        ON f.departure_airport=ap1.airport_code
        JOIN airports_data ap2
        ON f.arrival_airport=ap2.airport_code
        )
SELECT * 
FROM flight_data
WHERE actual_departure !='\\N' OR actual_arrival !='\\N'
LIMIT 10;

In [ ]:
# Top 10 Airports with the Most Delays
querry="""
WITH flight_data AS (
       SELECT 
                f.flight_id,
                f.flight_no,
                datetime(substr(f.scheduled_departure, 1, 19)) AS scheduled_departure,
                datetime(substr(f.actual_departure, 1, 19)) AS actual_departure,
                datetime(substr(f.scheduled_arrival, 1, 19)) AS scheduled_arrival,
                datetime(substr(f.actual_arrival, 1, 19)) AS actual_arrival,
                f.departure_airport,
                f.arrival_airport,
                f.status,
                json_extract(ap1.airport_name, '$.en') AS departure_airport_name,
                json_extract(ap1.city, '$.en') AS departure_city,
                ap1.coordinates AS departure_city_coordinates,
                ap1.timezone AS departure_timezone,
                json_extract(ap2.airport_name, '$.en') AS arrival_airport_name,
                json_extract(ap2.city, '$.en') AS arrival_city,
                ap2.coordinates AS arrival_city_coordinates,
                ap2.timezone AS arrival_timezone
        FROM flights f
        JOIN airports_data ap1
        ON f.departure_airport=ap1.airport_code
        JOIN airports_data ap2
        ON f.arrival_airport=ap2.airport_code
)

SELECT departure_airport_name, 
        ROUND(AVG(strftime('%s', datetime(actual_departure)) - strftime('%s', datetime(scheduled_departure)))/60,2) AS 'avg_delay (minutes)'
FROM flight_data
WHERE actual_departure !="\\N" OR actual_arrival !="\\N"
GROUP BY departure_airport_name
ORDER BY 2 DESC
LIMIT 10;
"""
df = pd.read_sql_query(querry, sqlite_conn)

# Plot
plt.figure(figsize=(10, 5))
sns.barplot(x=df['avg_delay (minutes)'], y=df['departure_airport_name'], palette='viridis')
plt.grid(axis='x', linestyle='--', alpha=0.6)

# Add labels and title
plt.xlabel('Average Delay (minutes)')
plt.ylabel('Airport Name')
plt.title('Top 10 Airports with Highest Departure Delays')

# Show the plot
plt.show();

In [ ]:
# Flight Status Distribution
querry="""
WITH flight_data AS (
       SELECT 
                f.flight_id,
                f.flight_no,
                datetime(substr(f.scheduled_departure, 1, 19)) AS scheduled_departure,
                datetime(substr(f.actual_departure, 1, 19)) AS actual_departure,
                datetime(substr(f.scheduled_arrival, 1, 19)) AS scheduled_arrival,
                datetime(substr(f.actual_arrival, 1, 19)) AS actual_arrival,
                f.departure_airport,
                f.arrival_airport,
                f.status,
                json_extract(ap1.airport_name, '$.en') AS departure_airport_name,
                json_extract(ap1.city, '$.en') AS departure_city,
                ap1.coordinates AS departure_city_coordinates,
                ap1.timezone AS departure_timezone,
                json_extract(ap2.airport_name, '$.en') AS arrival_airport_name,
                json_extract(ap2.city, '$.en') AS arrival_city,
                ap2.coordinates AS arrival_city_coordinates,
                ap2.timezone AS arrival_timezone
        FROM flights f
        JOIN airports_data ap1
        ON f.departure_airport=ap1.airport_code
        JOIN airports_data ap2
        ON f.arrival_airport=ap2.airport_code
        )

SELECT status, 
        COUNT(*) AS Counts
FROM flight_data
GROUP BY status
ORDER BY 2 DESC;
"""
df=pd.read_sql_query(querry, sqlite_conn)
plt.figure(figsize=(6, 6))
plt.pie(df['Counts'], labels=df['status'], 
        autopct='%1.1f%%', startangle=30, 
        colors=sns.color_palette('viridis', 5),
        wedgeprops=dict(width=0.6, edgecolor='w'),
        textprops={'fontsize': 11},
        shadow=True,
        pctdistance=0.85)

plt.title('Flight Status Distribution')
plt.show();

In [ ]:
# Top 10 Airports with the Most Delays
querry="""
WITH flight_data AS (
       SELECT 
                f.flight_id,
                f.flight_no,
                datetime(substr(f.scheduled_departure, 1, 19)) AS scheduled_departure,
                datetime(substr(f.actual_departure, 1, 19)) AS actual_departure,
                datetime(substr(f.scheduled_arrival, 1, 19)) AS scheduled_arrival,
                datetime(substr(f.actual_arrival, 1, 19)) AS actual_arrival,
                f.departure_airport,
                f.arrival_airport,
                f.status,
                json_extract(ap1.airport_name, '$.en') AS departure_airport_name,
                json_extract(ap1.city, '$.en') AS departure_city,
                ap1.coordinates AS departure_city_coordinates,
                ap1.timezone AS departure_timezone,
                json_extract(ap2.airport_name, '$.en') AS arrival_airport_name,
                json_extract(ap2.city, '$.en') AS arrival_city,
                ap2.coordinates AS arrival_city_coordinates,
                ap2.timezone AS arrival_timezone
        FROM flights f
        JOIN airports_data ap1
        ON f.departure_airport=ap1.airport_code
        JOIN airports_data ap2
        ON f.arrival_airport=ap2.airport_code
        )

SELECT flight_no, 
        ROUND(AVG(strftime('%s', datetime(actual_departure)) - strftime('%s', datetime(scheduled_departure)))/60,2) AS 'avg_delay (minutes)'
FROM flight_data
WHERE actual_departure !="\\N" OR actual_arrival !="\\N"
GROUP BY flight_no 
ORDER BY 2 DESC
LIMIT 10;
"""
dff=pd.read_sql_query(querry, sqlite_conn)
dff

In [ ]:
%%sql
# Most Popular Flight Hours
WITH flight_data AS (
       SELECT 
                f.flight_id,
                f.flight_no,
                datetime(substr(f.scheduled_departure, 1, 19)) AS scheduled_departure,
                datetime(substr(f.actual_departure, 1, 19)) AS actual_departure,
                datetime(substr(f.scheduled_arrival, 1, 19)) AS scheduled_arrival,
                datetime(substr(f.actual_arrival, 1, 19)) AS actual_arrival,
                f.departure_airport,
                f.arrival_airport,
                f.status,
                json_extract(ap1.airport_name, '$.en') AS departure_airport_name,
                json_extract(ap1.city, '$.en') AS departure_city,
                ap1.coordinates AS departure_city_coordinates,
                ap1.timezone AS departure_timezone,
                json_extract(ap2.airport_name, '$.en') AS arrival_airport_name,
                json_extract(ap2.city, '$.en') AS arrival_city,
                ap2.coordinates AS arrival_city_coordinates,
                ap2.timezone AS arrival_timezone
        FROM flights f
        JOIN airports_data ap1
        ON f.departure_airport=ap1.airport_code
        JOIN airports_data ap2
        ON f.arrival_airport=ap2.airport_code
        )

SELECT strftime('%HH', datetime(scheduled_departure)) AS departure_hour, 
       COUNT(*) AS flight_count
FROM flight_data
WHERE actual_departure !="\\N" OR actual_arrival !="\\N"
GROUP BY departure_hour
ORDER BY 2 DESC
LIMIT 10;

In [ ]:
# Most Popular Flight Hours
querry="""
WITH flight_data AS (
       SELECT 
                f.flight_id,
                f.flight_no,
                datetime(substr(f.scheduled_departure, 1, 19)) AS scheduled_departure,
                datetime(substr(f.actual_departure, 1, 19)) AS actual_departure,
                datetime(substr(f.scheduled_arrival, 1, 19)) AS scheduled_arrival,
                datetime(substr(f.actual_arrival, 1, 19)) AS actual_arrival,
                f.departure_airport,
                f.arrival_airport,
                f.status,
                json_extract(ap1.airport_name, '$.en') AS departure_airport_name,
                json_extract(ap1.city, '$.en') AS departure_city,
                ap1.coordinates AS departure_city_coordinates,
                ap1.timezone AS departure_timezone,
                json_extract(ap2.airport_name, '$.en') AS arrival_airport_name,
                json_extract(ap2.city, '$.en') AS arrival_city,
                ap2.coordinates AS arrival_city_coordinates,
                ap2.timezone AS arrival_timezone
        FROM flights f
        JOIN airports_data ap1
        ON f.departure_airport=ap1.airport_code
        JOIN airports_data ap2
        ON f.arrival_airport=ap2.airport_code
        )

SELECT strftime('%HH', datetime(scheduled_departure)) AS departure_hour, 
       COUNT(*) AS flight_count
FROM flight_data
WHERE actual_departure !="\\N" OR actual_arrival !="\\N"
GROUP BY departure_hour
ORDER BY 2 DESC
LIMIT 10;
"""
df_hour=pd.read_sql_query(querry, sqlite_conn)
df_hour


In [ ]:
%%sql
# Peak Travel Days
WITH flight_data AS (
       SELECT 
                f.flight_id,
                f.flight_no,
                datetime(substr(f.scheduled_departure, 1, 19)) AS scheduled_departure,
                datetime(substr(f.actual_departure, 1, 19)) AS actual_departure,
                datetime(substr(f.scheduled_arrival, 1, 19)) AS scheduled_arrival,
                datetime(substr(f.actual_arrival, 1, 19)) AS actual_arrival,
                f.departure_airport,
                f.arrival_airport,
                f.status,
                json_extract(ap1.airport_name, '$.en') AS departure_airport_name,
                json_extract(ap1.city, '$.en') AS departure_city,
                ap1.coordinates AS departure_city_coordinates,
                ap1.timezone AS departure_timezone,
                json_extract(ap2.airport_name, '$.en') AS arrival_airport_name,
                json_extract(ap2.city, '$.en') AS arrival_city,
                ap2.coordinates AS arrival_city_coordinates,
                ap2.timezone AS arrival_timezone
        FROM flights f
        JOIN airports_data ap1
        ON f.departure_airport=ap1.airport_code
        JOIN airports_data ap2
        ON f.arrival_airport=ap2.airport_code
        )

SELECT CASE strftime('%w', scheduled_departure)
        WHEN '0' THEN 'Sunday'
        WHEN '1' THEN 'Monday'
        WHEN '2' THEN 'Tuesday'
        WHEN '3' THEN 'Wednesday'
        WHEN '4' THEN 'Thursday'
        WHEN '5' THEN 'Friday'
        WHEN '6' THEN 'Saturday'
    END AS flight_weekday,
       COUNT(*) AS total_flights
FROM flight_data
WHERE actual_departure !="\\N" OR actual_arrival !="\\N"
GROUP BY flight_weekday
ORDER BY 2 DESC
LIMIT 10;

In [ ]:
# Peak Travel Days
querry="""
WITH flight_data AS (
       SELECT 
                f.flight_id,
                f.flight_no,
                datetime(substr(f.scheduled_departure, 1, 19)) AS scheduled_departure,
                datetime(substr(f.actual_departure, 1, 19)) AS actual_departure,
                datetime(substr(f.scheduled_arrival, 1, 19)) AS scheduled_arrival,
                datetime(substr(f.actual_arrival, 1, 19)) AS actual_arrival,
                f.departure_airport,
                f.arrival_airport,
                f.status,
                json_extract(ap1.airport_name, '$.en') AS departure_airport_name,
                json_extract(ap1.city, '$.en') AS departure_city,
                ap1.coordinates AS departure_city_coordinates,
                ap1.timezone AS departure_timezone,
                json_extract(ap2.airport_name, '$.en') AS arrival_airport_name,
                json_extract(ap2.city, '$.en') AS arrival_city,
                ap2.coordinates AS arrival_city_coordinates,
                ap2.timezone AS arrival_timezone
        FROM flights f
        JOIN airports_data ap1
        ON f.departure_airport=ap1.airport_code
        JOIN airports_data ap2
        ON f.arrival_airport=ap2.airport_code
        )

SELECT CASE strftime('%w', scheduled_departure)
        WHEN '0' THEN 'Sunday'
        WHEN '1' THEN 'Monday'
        WHEN '2' THEN 'Tuesday'
        WHEN '3' THEN 'Wednesday'
        WHEN '4' THEN 'Thursday'
        WHEN '5' THEN 'Friday'
        WHEN '6' THEN 'Saturday'
    END AS flight_weekday,
       COUNT(*) AS total_flights
FROM flight_data
WHERE actual_departure !="\\N" OR actual_arrival !="\\N"
GROUP BY flight_weekday
ORDER BY 2 DESC
LIMIT 10;
"""
df_days=pd.read_sql_query(querry, sqlite_conn)
df_days